# 07 — R3 Counterparty Bot-Trade Tape EDA

Characterize the R3 counterparty trade tape and identify exploitable bot patterns.
UCSD's R5 *Olivia* trick (per-bot edge ranking) requires named buyer/seller IDs — we verify whether the R3 tape exposes them.

**Inputs:** `data/round_3/trades_round_3_day_{0,1,2}.csv`, `prices_round_3_day_*.csv`.
**Outputs:** plots in `docs/round_3/research/plots/05_*.png`, summary CSV `docs/round_3/research/05_bot_trades_summary.csv`.

Run all cells top-to-bottom. Final cell is a TL;DR summary of every finding.

## Setup

Imports, paths, constants. `%matplotlib inline` so plots render in-notebook.

In [ ]:
%matplotlib inline
from __future__ import annotations
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/Users/bensinek/Documents/Coding/Prosperity4')
DATA = ROOT / "data" / "round_3"
PLOTS = ROOT / "docs" / "round_3" / "research" / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

DAYS = [0, 1, 2]
TICK_MS = 100  # 1 timestamp unit = 1 game tick (per IMC convention)

## Load trade tape and price book

Pool all 3 days of trades and price snapshots. Both CSVs are semicolon-separated.

In [ ]:
# ---------- load ----------
def load_trades() -> pd.DataFrame:
    frames = []
    for d in DAYS:
        df = pd.read_csv(DATA / f"trades_round_3_day_{d}.csv", sep=";")
        df["day"] = d
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def load_prices() -> pd.DataFrame:
    frames = []
    for d in DAYS:
        df = pd.read_csv(DATA / f"prices_round_3_day_{d}.csv", sep=";")
        # day already in file
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


trades = load_trades()
prices = load_prices()
print(f"trades: {len(trades):,}   prices: {len(prices):,}")

## Q1 — Tape characterization

Trades per day, trades per product, and which products have **zero** counterparty tape.

In [ ]:
# ---------- Q1: tape characterization ----------
per_day = trades.groupby("day").size().rename("n_trades")
per_prod = trades.groupby("symbol").agg(n_trades=("price", "size"),
                                        n_volume=("quantity", "sum"))
per_prod = per_prod.sort_values("n_trades", ascending=False)
print("\n=== Q1: trades per day ===\n", per_day)
print("\n=== Q1: trades per product (all 3 days pooled) ===\n", per_prod)

all_products = sorted(prices["product"].unique())
dead = [p for p in all_products if p not in per_prod.index]
print(f"\nProducts in price feed: {len(all_products)} -> {all_products}")
print(f"Products with ZERO trades: {dead}")

## Q2 — Counterparty IDs

Check whether `buyer` / `seller` columns are populated. If anonymized, UCSD-style per-bot ranking is impossible in R3.

In [ ]:
# ---------- Q2: buyer/seller IDs ----------
n_buyer = trades["buyer"].notna().sum() if "buyer" in trades.columns else 0
n_seller = trades["seller"].notna().sum() if "seller" in trades.columns else 0
print(f"\n=== Q2: counterparty IDs ===")
print(f"buyer non-null: {n_buyer} / {len(trades)}")
print(f"seller non-null: {n_seller} / {len(trades)}")
if n_buyer == 0 and n_seller == 0:
    print(">>> R3 tape is FULLY ANONYMIZED. UCSD-style per-bot ranking N/A. "
          "Defer to R5.")

## Q3 — Trade-size distribution per product

Lot-size summary plus per-product histogram. Fixed-lot quirks hint at single-bot quoters.

In [ ]:
# ---------- Q3: trade-size distribution per product ----------
size_summary = trades.groupby("symbol")["quantity"].describe()[
    ["count", "mean", "50%", "max"]
].rename(columns={"50%": "median"})
print("\n=== Q3: trade size summary ===\n", size_summary)

# per-product histogram
products_with_trades = list(per_prod.index)
n = len(products_with_trades)
fig, axes = plt.subplots((n + 2) // 3, 3, figsize=(14, 3 * ((n + 2) // 3)))
for i, prod in enumerate(products_with_trades):
    ax = axes.flat[i]
    q = trades[trades["symbol"] == prod]["quantity"]
    ax.hist(q, bins=range(1, int(q.max()) + 2), color="steelblue",
            edgecolor="black")
    ax.set_title(f"{prod} (n={len(q)})")
    ax.set_xlabel("qty")
for j in range(i + 1, len(axes.flat)):
    axes.flat[j].axis("off")
fig.tight_layout()
fig.savefig(PLOTS / "05_trade_size_hist.png", dpi=110)
plt.close(fig)

## Attach mid / bid / ask to each trade

Join every trade to the price book at the same `(day, timestamp, product)`. Needed for Lee-Ready aggressor classification and post-trade drift.

In [ ]:
# ---------- build mid-price lookup for join ----------
mid_lkp = prices.set_index(["day", "timestamp", "product"])[
    ["mid_price", "bid_price_1", "ask_price_1"]
]


def attach_mid(tr: pd.DataFrame) -> pd.DataFrame:
    """Join trades to price book at same (day, timestamp, product)."""
    tr = tr.copy()
    keys = list(zip(tr["day"], tr["timestamp"], tr["symbol"]))
    out = mid_lkp.reindex(keys).reset_index(drop=True)
    tr["mid"] = out["mid_price"].values
    tr["bid"] = out["bid_price_1"].values
    tr["ask"] = out["ask_price_1"].values
    return tr


trades = attach_mid(trades)

## Q4 — Aggressor inference (Lee-Ready)

Classify each trade as buy-aggressor (lifted offer), sell-aggressor (hit bid), or mid-price using the standard Lee-Ready tick rule with mid fallback.

In [ ]:
# ---------- Q4: aggressor inference (Lee-Ready style) ----------
def classify(row):
    p, b, a = row["price"], row["bid"], row["ask"]
    if pd.isna(b) or pd.isna(a):
        return "unknown"
    if p >= a:
        return "buy_aggr"   # buyer lifted offer
    if p <= b:
        return "sell_aggr"  # seller hit bid
    # tick rule fallback: compare to mid
    if not pd.isna(row["mid"]):
        if p > row["mid"]:
            return "buy_aggr"
        if p < row["mid"]:
            return "sell_aggr"
    return "mid"


trades["side"] = trades.apply(classify, axis=1)
side_mat = trades.groupby(["symbol", "side"]).size().unstack(fill_value=0)
side_mat["n"] = side_mat.sum(axis=1)
for c in ["buy_aggr", "sell_aggr"]:
    if c in side_mat.columns:
        side_mat[f"%{c}"] = (side_mat[c] / side_mat["n"] * 100).round(1)
print("\n=== Q4: aggressor side per product ===\n", side_mat)

## Q5 — Intraday timing

Bucket trades into 100k-timestamp windows (100 buckets/day) and look for clustering.

In [ ]:
# ---------- Q5: intraday timing ----------
trades["bucket"] = (trades["timestamp"] // 100_000).astype(int)  # 100 buckets/day
fig, ax = plt.subplots(figsize=(11, 4))
for prod, grp in trades.groupby("symbol"):
    if len(grp) < 30:
        continue
    h = grp["bucket"].value_counts().sort_index()
    ax.plot(h.index, h.values, marker=".", label=prod, alpha=0.7)
ax.set_xlabel("timestamp bucket (100k units)")
ax.set_ylabel("trades per bucket (pooled across 3 days)")
ax.set_title("Intraday trade clustering")
ax.legend(fontsize=7, ncol=3)
fig.tight_layout()
fig.savefig(PLOTS / "05_intraday_timing.png", dpi=110)
plt.close(fig)

## Q6 — Mid drift after trade

For each trade, compute mid change at +5, +10, +50 ticks. Then sign by aggressor: positive signed drift means the aggressor was **right** (informed flow) — negative means **wrong** (dumb flow we should fade).

In [ ]:
# ---------- Q6: mid drift after trade ----------
# Build per-(day,product) fast mid lookup keyed by tick index.
mid_by_dp: dict[tuple[int, str], pd.Series] = {}
for (d, prod), g in prices.groupby(["day", "product"]):
    s = g.sort_values("timestamp").set_index("timestamp")["mid_price"]
    mid_by_dp[(d, prod)] = s


def drift(row, k: int) -> float:
    s = mid_by_dp.get((row["day"], row["symbol"]))
    if s is None or pd.isna(row["mid"]):
        return np.nan
    target_ts = row["timestamp"] + k * 100  # 1 tick = 100 timestamp units
    # nearest ts >= target
    idx = s.index.searchsorted(target_ts)
    if idx >= len(s):
        return np.nan
    fut = s.iloc[idx]
    return fut - row["mid"]


for k in (5, 10, 50):
    trades[f"drift_{k}"] = trades.apply(lambda r, k=k: drift(r, k), axis=1)

# signed drift: positive when aligned with aggressor (buyer-aggressor profits if mid up)
def signed(row, k):
    d = row[f"drift_{k}"]
    if pd.isna(d):
        return np.nan
    if row["side"] == "buy_aggr":
        return d
    if row["side"] == "sell_aggr":
        return -d
    return np.nan


for k in (5, 10, 50):
    trades[f"signed_drift_{k}"] = trades.apply(lambda r, k=k: signed(r, k),
                                                axis=1)

drift_summary = (
    trades.groupby("symbol")[
        ["signed_drift_5", "signed_drift_10", "signed_drift_50"]
    ].mean().round(3)
)
drift_summary["n_classified"] = trades.groupby("symbol")[
    "signed_drift_5"
].apply(lambda x: x.notna().sum())
print("\n=== Q6: signed mid-drift after trade (aggressor-aligned) ===")
print("Positive => aggressor was 'right' (informed flow). "
      "Negative => aggressor was 'wrong' (dumb flow → fade).")
print(drift_summary)

# plot signed drift bar per product
fig, ax = plt.subplots(figsize=(11, 4))
ds = drift_summary.dropna(subset=["signed_drift_10"]).sort_values(
    "signed_drift_10"
)
ax.barh(ds.index, ds["signed_drift_10"], color=[
    "tab:red" if v < 0 else "tab:green" for v in ds["signed_drift_10"]
])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("signed mid drift @ +10 ticks (aggressor pov)")
ax.set_title("Aggressor edge per product (>0 = follow aggressor; <0 = fade)")
fig.tight_layout()
fig.savefig(PLOTS / "05_aggressor_drift.png", dpi=110)
plt.close(fig)

## Q7 — VEV_6000 / VEV_6500 zero-price ghost trades

Deep-OTM voucher tape consists entirely of price-0 fills. Check whether VEV_6000 and VEV_6500 zero-trades are **paired** at the same timestamp + quantity, and whether the underlying mid is frozen.

In [ ]:
# ---------- Q7: VEV_6000 / VEV_6500 zero-price trades ----------
zero = trades[(trades["symbol"].isin(["VEV_6000", "VEV_6500"])) &
              (trades["price"] == 0.0)]
print("\n=== Q7: zero-price OTM trades ===")
print(f"Total zero-price trades: {len(zero)}")
print("Per-symbol & per-day count:")
print(zero.groupby(["day", "symbol"]).size())

# pair check: do VEV_6000 and VEV_6500 occur at SAME (day, timestamp, qty)?
pairs = zero.groupby(["day", "timestamp"]).agg(
    syms=("symbol", lambda s: tuple(sorted(s.unique()))),
    qtys=("quantity", lambda q: tuple(sorted(q.unique()))),
)
n_paired = (pairs["syms"] == ("VEV_6000", "VEV_6500")).sum()
print(f"Joint (6000+6500) timestamps: {n_paired} / {len(pairs)}")
matched_qty = pairs.apply(lambda r: len(r["qtys"]) == 1, axis=1).sum()
print(f"Same-quantity matches: {matched_qty} / {len(pairs)}")
# also check all VEV_6000 prices nonzero?
vev6 = trades[trades["symbol"].isin(["VEV_6000", "VEV_6500"])]
print(f"All VEV_6000/6500 trades at price 0? "
      f"{(vev6['price'] == 0).all()}")
# voucher mid at those times
for d in DAYS:
    s6000 = mid_by_dp.get((d, "VEV_6000"))
    s6500 = mid_by_dp.get((d, "VEV_6500"))
    if s6000 is not None:
        print(f"  day {d} VEV_6000 mid range: "
              f"{s6000.min()}..{s6000.max()}  "
              f"VEV_6500 mid range: {s6500.min()}..{s6500.max()}")

## Q8 — Counterparty edge ranking (skipped)

Requires named buyer/seller IDs. R3 tape is anonymized — defer to R5.

In [ ]:
# ---------- Q8: counterparty edge ranking ----------
print("\n=== Q8: counterparty edge ranking ===")
print("SKIPPED — buyer/seller IDs anonymized in R3 tape. "
      "Methodology applicable in R5 if IDs leak there.")

## Q9 — Voucher-bot patterns

Per-strike side breakdown across the VEV chain. Useful to see whether OTM strikes are dump-only (sell-aggressor only) vs two-sided.

In [ ]:
# ---------- Q9: voucher-bot patterns ----------
voucher_breakdown = (
    trades[trades["symbol"].str.startswith("VEV_")]
    .groupby(["symbol", "side"]).size().unstack(fill_value=0)
)
voucher_breakdown["n"] = voucher_breakdown.sum(axis=1)
print("\n=== Q9: voucher trade activity by side ===\n", voucher_breakdown)

## Save findings table

Join drift + side stats into a per-product summary CSV for the docs folder.

In [ ]:
# ---------- save findings table ----------
out = drift_summary.join(side_mat[[c for c in side_mat.columns if c != "n"]],
                         how="outer")
out.to_csv(PLOTS.parent / "05_bot_trades_summary.csv")
print(f"\nWrote {PLOTS.parent / '05_bot_trades_summary.csv'}")

## Summary + Key Findings

30-second TL;DR — every observation captured.

### Counterparty IDs
- `buyer` / `seller` columns are **100% null** across all 3 days, all 4,048 trades.
- R3 tape is **fully anonymized** — UCSD's *Olivia*-style per-bot edge ranking is **impossible in R3**.
- Re-run this notebook on R5 trade tape immediately when it drops; if names appear, the Q8 methodology becomes directly applicable.

### Tape volume (3 days pooled, ~1300-1400 trades/day)
- VELVETFRUIT_EXTRACT — 1,372 trades / 8,269 vol, 43% sell_aggr (most active, ~33% of tape)
- HYDROGEL_PACK — 1,010 trades / 4,078 vol, 48% sell_aggr
- VEV_4000 — 464 trades / 940 vol, 51% sell_aggr (deep ITM, behaves like delta-1 underlying)
- VEV_6000 — 284 trades / 1,002 vol, 100% sell_aggr (price=0)
- VEV_6500 — 284 trades / 1,002 vol, 100% sell_aggr (price=0)
- VEV_5500 — 267 trades / 937 vol, 100% sell_aggr
- VEV_5400 — 225 trades / 787 vol, 100% sell_aggr
- VEV_5300 — 121 trades / 420 vol, 98% sell_aggr
- VEV_5200 — 18 trades / 63 vol, 94% sell_aggr (very thin)
- VEV_4500 / VEV_5000 / VEV_5100 — **1 trade each** → effectively dead in tape

### Trade-size distribution
- HYDROGEL: lots 1-6, mode 4 → fixed-lot bot.
- VFE: lots 1-15, mode 6.
- All vouchers: lots 1-5, mode 4 — same lot profile across OTM strikes → likely **same bot family** quoting the OTM chain.
- No single round-number size dominates >40% in any product.

### Aggressor balance
- HYDROGEL/VFE/VEV_4000 are roughly symmetric two-sided — consistent with anonymized 2-sided market-making bots; no one-way directional flow.
- OTM vouchers (5300/5400/5500/6000/6500) are **100% sell-aggressor** (98% at 5300) — bots only DUMP OTM vouchers, never lift offers.

### Mid-drift after trade — the ONE actionable signal
- **VFE buy_aggr**: drift @ +50 ticks = **+0.629**, n=780, std 6.66, **t = +2.64** → statistically significant. Bots BUYING VFE are informed.
- VFE sell_aggr: drift = +0.077, t = +0.29 → flat, not informed.
- VEV_4000 buy_aggr: -0.011, t = -0.03 / sell_aggr: -0.323, t = -0.68 → noise.
- VEV_5300 sell_aggr: -0.340, t = -1.40 → slightly informed, sub-significance.
- VEV_5400 sell_aggr: -0.036, t = -0.39.
- VEV_5500 sell_aggr: +0.009, t = +0.23.
- HYDROGEL buy_aggr: -0.178, t = -0.31 / sell_aggr: +0.727, t = +1.23.
- **Only VFE buy_aggr crosses statistical significance**; all others |t|<1.5.
- Implication: lean LONG VFE on bot buy bursts; ignore VFE sell aggression.

### VEV_6000 / VEV_6500 zero-price ghost trades
- 568 total zero-price trades (284 in each strike).
- **100% paired**: every zero-price (day, ts) has both VEV_6000 AND VEV_6500 with identical quantity.
- Mid for both symbols **pinned at 0.5** every tick of every day.
- ALL VEV_6000/6500 trades in entire tape are at price 0 (no other price ever traded).
- Interpretation: tape-only matched-pair **ghost events** — almost certainly platform settlement / paired-quote self-cross at deep-OTM zero-bid books, or end-of-tick voucher liquidation accounting.
- Order book NEVER actually shows a 0-priced ask we can lift. **Not free money. Do NOT try to lift VEV_6000/6500 at 0.**

### Voucher-bot pattern
- Trade activity concentrates at OTM strikes (5300/5400/5500) and deep-ITM (4000); ATM (4500/5000/5100) bots absent.
- OTM dumps not informed (signed drift ~0) → **passive systematic vega / short-premium quoters**, not predictive.
- We can sit on the BID at OTM strikes and harvest these dumps cheaply.

### Per-product flow toxicity (for OUR market-making)
- **HYDROGEL_PACK** — **LOW**. Two-sided (52/48), no informed drift, wide stable spread. Safe MM target.
- **VELVETFRUIT_EXTRACT** — **MEDIUM**. Two-sided but buy-aggressor informed (+0.63t, t=+2.64). Risk: tight ASK gets adversely selected. Quote asymmetric (wider ask than bid) OR lean inventory long. Also: VFE is voucher underlying → voucher-team hedging may add unseen toxicity.
- **VEV_4000** — **LOW-MEDIUM**. Behaves delta-1 (deep ITM); no informed flow.
- **VEV_5300 / 5400 / 5500** — **MEDIUM-LOW**. Bots only sell. Sit on BID inside their dump price → harvest cheap inventory. Short-vega positions; need vol model.
- **VEV_4500 / 5000 / 5100** — **N/A**. No counterparty tape (1 trade each). Use for arb against other strikes; don't expect counterparty fills.
- **VEV_5200** — **N/A** (18 trades, thin).
- **VEV_6000 / VEV_6500** — **DEAD**. Mid pinned 0.5; only 0-price ghost trades. Do not trade.

### Anonymization caveat
- All conclusions are **per-product aggregate** — no per-bot decomposition possible in R3.
- Two distinct bot families could net to the observed signed drift; can't tell apart.
- If R5 leaks names, re-run Q8 to rank counterparties.

### Actionable items
1. **VFE buy-aggression signal**: count buyer-aggressor trades (or signed volume) in last K ticks; bias inventory long when it spikes. Expect ~0.6t edge over 50-tick horizon.
2. **OTM voucher MM (5300-5500)**: post passive bids ≤ wall_mid; expect to be filled by dumping bots; mark short positions with own vol model (real options, IV/skew matter).
3. **Skip VEV_6000/6500** entirely.
4. Re-run on R5 tape ASAP — if names appear, paste the Q8 counterparty-ranking block.